In [1]:
import sys
import logging
from pathlib import Path

import torch

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from utils.io_utils import load_fold
from utils.mlp_utils import (
    set_seed,
    prepare_all_fold_tensors,
    run_nested_random_search,
    print_final_results,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

logger = logging.getLogger(__name__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
logger.info(f"Device: {device}")

GLOBAL_SEED = 42
set_seed(GLOBAL_SEED)

01:27:09 [INFO] Device: cuda


In [2]:
CFG = {
    "task": "hi",
    "dataset": "sol",
    "fp_type": "rdkit_desc",
    "n_bits": 1024,

    "outer_folds": [1, 2, 3],

    "inner_k": 2,

    "random_state": GLOBAL_SEED,
}

In [3]:
SEARCH_SPACE = {
    "n_layers":     [1, 2, 3],
    "n_nodes":      [64, 128, 256, 512, 1024],
    "r":            [0.5, 0.7, 1.0],
    "activation":   ["relu", "leaky_relu", "gelu", "silu"],  
    "dropout":      [0.0, 0.2, 0.3, 0.5],
    "batchnorm":    [True, False],
    "init":         ["kaiming", "xavier"],

    "lr":           [1e-4, 5e-4, 1e-3, 2e-3, 3e-3],
    "weight_decay": [0.0, 1e-5, 1e-4, 1e-3, 5e-3, 1e-2],
    "batch_size":   [64, 128, 256],
}

FIXED_HP = {
    "max_epochs": 100,
    "patience":   10,
    "grad_clip":  1.0,
}

N_ITER  = 150
N_SEEDS = 3

In [4]:
folds_data = {}

for fold_idx in CFG["outer_folds"]:
    train_df, test_df = load_fold(
        CFG["task"],
        CFG["dataset"],
        fold_idx,
    )

    folds_data[fold_idx] = {
        "train": train_df,
        "test": test_df,
    }

    n_train = len(train_df)
    n_test = len(test_df)

    n_train_pos = int(train_df["value"].sum())
    n_train_neg = n_train - n_train_pos

    n_test_pos = int(test_df["value"].sum())
    n_test_neg = n_test - n_test_pos

    logger.info(
        f"Fold {fold_idx} | "
        f"train={n_train} "
        f"(pos={n_train_pos}, neg={n_train_neg}, "
        f"pos_ratio={n_train_pos / n_train:.3f}) | "
        f"test={n_test} "
        f"(pos={n_test_pos}, neg={n_test_neg}, "
        f"pos_ratio={n_test_pos / n_test:.3f})"
    )

01:27:09 [INFO] Loaded hi/sol fold 1: train=1442, test=721
01:27:09 [INFO] Fold 1 | train=1442 (pos=308, neg=1134, pos_ratio=0.214) | test=721 (pos=158, neg=563, pos_ratio=0.219)
01:27:09 [INFO] Loaded hi/sol fold 2: train=1442, test=721
01:27:09 [INFO] Fold 2 | train=1442 (pos=305, neg=1137, pos_ratio=0.212) | test=721 (pos=161, neg=560, pos_ratio=0.223)
01:27:09 [INFO] Loaded hi/sol fold 3: train=1442, test=721
01:27:09 [INFO] Fold 3 | train=1442 (pos=319, neg=1123, pos_ratio=0.221) | test=721 (pos=147, neg=574, pos_ratio=0.204)


In [5]:
from utils.mlp_utils import prepare_all_fold_tensors

folds_tensors = prepare_all_fold_tensors(
    cfg=CFG,
    folds_data=folds_data,
    logger=logger,
)

01:27:09 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/sol/rdkit_desc_train_1.npz
01:27:09 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/sol/rdkit_desc_test_1.npz
01:27:09 [INFO] Fold 1 | X_train: (1442, 217), X_test: (721, 217) | scale_features=True | pos_weight=3.682
01:27:09 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/sol/rdkit_desc_train_2.npz
01:27:09 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/sol/rdkit_desc_test_2.npz
01:27:09 [INFO] Fold 2 | X_train: (1442, 217), X_test: (721, 217) | scale_features=True | pos_weight=3.728
01:27:09 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/sol/rdkit_desc_train_3.npz
01:27:09 [INFO] Loading fingerprints from cache: /home/f.capria/drug-discovery-lohi/features/hi/sol/rdkit_desc_test_3.npz
01:27:09 [INFO] Fold 3 | X_train: (1442, 217), X_te

In [6]:
logger.info(f"Estimated time: ~{N_ITER * 6 * len(CFG['outer_folds']) / 60:.0f} min")

fold_results = run_nested_random_search(
    cfg=CFG,
    folds_tensors=folds_tensors,
    search_space=SEARCH_SPACE,
    fixed_hp=FIXED_HP,
    n_iter=N_ITER,
    n_seeds=N_SEEDS,
    device=device,
    seed=GLOBAL_SEED,
    logger=logger,
)

print_final_results(fold_results, title="MLP sol Hi — rdkit_desc")

01:27:09 [INFO] Estimated time: ~45 min
01:27:09 [INFO] 
OUTER FOLD 1
01:27:12 [INFO]   [1/150] inner PR-AUC=0.5863 (3s) | L=2 N=512 r=0.7 dropout=0.3 lr=3e-03
01:27:14 [INFO]   [2/150] inner PR-AUC=0.5839 (2s) | L=3 N=128 r=1.0 dropout=0.2 lr=1e-03
01:27:16 [INFO]   [3/150] inner PR-AUC=0.5659 (2s) | L=1 N=1024 r=0.7 dropout=0.0 lr=5e-04
01:27:18 [INFO]   [4/150] inner PR-AUC=0.5797 (2s) | L=2 N=64 r=0.7 dropout=0.3 lr=3e-03
01:27:19 [INFO]   [5/150] inner PR-AUC=0.5655 (1s) | L=2 N=256 r=1.0 dropout=0.5 lr=3e-03
01:27:26 [INFO]   [6/150] inner PR-AUC=0.5441 (7s) | L=1 N=64 r=0.7 dropout=0.5 lr=1e-04
01:27:29 [INFO]   [7/150] inner PR-AUC=0.5913 (3s) | L=3 N=128 r=1.0 dropout=0.3 lr=1e-03
01:27:34 [INFO]   [8/150] inner PR-AUC=0.5684 (5s) | L=2 N=64 r=0.7 dropout=0.5 lr=5e-04
01:27:37 [INFO]   [9/150] inner PR-AUC=0.5578 (4s) | L=1 N=256 r=0.5 dropout=0.3 lr=1e-04
01:27:42 [INFO]   [10/150] inner PR-AUC=0.5342 (5s) | L=3 N=64 r=0.5 dropout=0.0 lr=5e-04
01:27:53 [INFO]   [11/150] inner


MLP sol Hi — rdkit_desc
Fold 1: PR-AUC=0.5941  ROC-AUC=0.8561
Fold 2: PR-AUC=0.6305  ROC-AUC=0.8511
Fold 3: PR-AUC=0.5669  ROC-AUC=0.8431

Aggregated metrics:
  pr_auc_mean: 0.5972
  pr_auc_std: 0.0261
  roc_auc_mean: 0.8501
  roc_auc_std: 0.0054
  bedroc_mean: 0.7333
  bedroc_std: 0.0744
  f1_at_05_mean: 0.5964
  f1_at_05_std: 0.0094
  positive_rate_mean: 0.2154
  positive_rate_std: 0.0083

Best hyperparameters per fold:
Fold 1: L=2 N=128 r=1.0 act=relu dropout=0.5 bn=False init=kaiming lr=5e-04 wd=1e-04 bs=64
Fold 2: L=1 N=256 r=0.7 act=gelu dropout=0.2 bn=True init=kaiming lr=3e-03 wd=1e-02 bs=64
Fold 3: L=1 N=512 r=0.7 act=relu dropout=0.5 bn=True init=xavier lr=1e-04 wd=1e-03 bs=64


{'pr_auc_mean': np.float64(0.5972),
 'pr_auc_std': np.float64(0.0261),
 'roc_auc_mean': np.float64(0.8501),
 'roc_auc_std': np.float64(0.0054),
 'bedroc_mean': np.float64(0.7333),
 'bedroc_std': np.float64(0.0744),
 'f1_at_05_mean': np.float64(0.5964),
 'f1_at_05_std': np.float64(0.0094),
 'positive_rate_mean': np.float64(0.2154),
 'positive_rate_std': np.float64(0.0083)}

In [7]:
import json
from pathlib import Path

out_dir = PROJECT_ROOT / "results" / CFG["task"] / CFG["dataset"] / f"mlp_{CFG['dataset']}_{CFG['task']}_{CFG['fp_type']}"
out_dir.mkdir(parents=True, exist_ok=True)

for res in fold_results:
    fold = res["fold"]

    with open(out_dir / f"params_fold_{fold}.json", "w") as f:
        json.dump({
            "fold": fold,
            "best_params": res["best_hp"],
            "inner_cv_score": res["inner_score"],
            "inner_train_loss_diagnostic": res["best_train_loss_diagnostic"],
            "test_metrics": res["test_metrics"],
            "seed_results": res["seed_results"],
        }, f, indent=2, default=str)

print(f"Saved JSON files in: {out_dir}")

Saved JSON files in: /home/f.capria/drug-discovery-lohi/results/hi/sol/mlp_sol_hi_rdkit_desc
